In [10]:
import pandas as pd

# Load datasets for comparison
# Our curated dataset
df_ours = pd.read_csv('/Users/francisco/Library/CloudStorage/OneDrive-Pessoal/Documentos/LabMol/SOFIA/MQ/GitHub OdorSight/OdorSight/Curation/curated_dataset_auto_inspection.csv')
# Benchmark dataset (Odorify)
df_benchmark = pd.read_excel('/Users/francisco/Library/CloudStorage/OneDrive-Pessoal/Documentos/LabMol/SOFIA/MQ/GitHub OdorSight/OdorSight/benchmark_odorify/scripts/odorify_dataset.xlsx')


In [11]:
from rdkit import Chem

# Function to compute InChIKey from SMILES
def smiles_to_inchikey(smiles):
    """
    Convert SMILES string to InChIKey for robust molecular comparison.
    InChIKey provides a standardized identifier independent of SMILES representation.
    
    Args:
        smiles (str): SMILES string representation of the molecule
    
    Returns:
        str or None: InChIKey string if successful, None otherwise
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            return Chem.MolToInchiKey(mol)
    except:
        pass
    return None

# Compute InChIKey for both datasets
print("Computing InChIKeys for molecular standardization...")
df_ours['InChIKey'] = df_ours['SMILES'].apply(smiles_to_inchikey)
df_benchmark['InChIKey'] = df_benchmark['SMILES'].apply(smiles_to_inchikey)

# Remove invalid entries (None/NaN)
df_ours_clean = df_ours.dropna(subset=['InChIKey'])
df_benchmark_clean = df_benchmark.dropna(subset=['InChIKey'])

# Extract unique InChIKey sets
inchikey_ours = set(df_ours_clean['InChIKey'])
inchikey_benchmark = set(df_benchmark_clean['InChIKey'])

# Calculate overlap (intersection)
overlap = inchikey_ours.intersection(inchikey_benchmark)

# Calculate statistics
total_ours = len(inchikey_ours)
total_benchmark = len(inchikey_benchmark)
total_overlap = len(overlap)

percentage_ours = (total_overlap / total_ours * 100) if total_ours > 0 else 0
percentage_benchmark = (total_overlap / total_benchmark * 100) if total_benchmark > 0 else 0

# Display results
print(f"\nDataset Overlap Analysis:")
print(f"  Unique InChIKeys in our dataset: {total_ours}")
print(f"  Unique InChIKeys in benchmark dataset: {total_benchmark}")
print(f"  Total overlapping InChIKeys: {total_overlap}")
print(f"\nOverlap percentages:")
print(f"  Relative to our dataset: {percentage_ours:.2f}%")
print(f"  Relative to benchmark dataset: {percentage_benchmark:.2f}%")

# Compute exclusive compounds
exclusive_ours = inchikey_ours - inchikey_benchmark
exclusive_benchmark = inchikey_benchmark - inchikey_ours

print(f"\nExclusive compounds:")
print(f"  Unique to our dataset: {len(exclusive_ours)}")
print(f"  Unique to benchmark dataset: {len(exclusive_benchmark)}")


Computing InChIKeys for molecular standardization...


[10:21:40] Explicit valence for atom # 0 N, 6, is greater than permitted



Dataset Overlap Analysis:
  Unique InChIKeys in our dataset: 4212
  Unique InChIKeys in benchmark dataset: 5831
  Total overlapping InChIKeys: 3192

Overlap percentages:
  Relative to our dataset: 75.78%
  Relative to benchmark dataset: 54.74%

Exclusive compounds:
  Unique to our dataset: 1020
  Unique to benchmark dataset: 2639


[10:21:40] Explicit valence for atom # 0 Cl, 7, is greater than permitted
[10:21:40] Explicit valence for atom # 1 Br, 5, is greater than permitted
[10:21:40] Explicit valence for atom # 1 Cl, 3, is greater than permitted
[10:21:40] WARNING: not removing hydrogen atom without neighbors
[10:21:40] Explicit valence for atom # 0 Cl, 4, is greater than permitted
[10:21:40] WARNING: not removing hydrogen atom without neighbors


In [12]:
from sklearn.model_selection import train_test_split


# Stratified train-test split with benchmark dataset leakage prevention
# Objective: Create a 90:10 train-test split ensuring the test set contains
# only molecules that are NOT present in the benchmark dataset


# Identify molecules in our dataset that overlap with benchmark
df_ours_clean['no_overlap'] = ~df_ours_clean['InChIKey'].isin(overlap)


# Partition molecules into two groups:
# 1. Exclusive molecules (not in benchmark) - eligible for both train and test
# 2. Overlapping molecules (in benchmark) - restricted to training set only
df_exclusive = df_ours_clean[df_ours_clean['no_overlap']].copy()
df_overlapping = df_ours_clean[~df_ours_clean['no_overlap']].copy()


print(f"Dataset Partitioning Analysis:")
print(f"  Total molecules in our dataset: {len(df_ours_clean)}")
print(f"  Exclusive molecules (not in benchmark): {len(df_exclusive)}")
print(f"  Overlapping molecules (present in benchmark): {len(df_overlapping)}")


# Calculate test set size (10% of total dataset)
total_size = len(df_ours_clean)
test_size = int(total_size * 0.10)


print(f"\nTrain-Test Split Configuration:")
print(f"  Target test set size (10%): {test_size} molecules")
print(f"  Exclusive molecules available: {len(df_exclusive)}")


# Validate sufficient exclusive molecules for test set
if len(df_exclusive) < test_size:
    print(f"\nWARNING: Insufficient exclusive molecules for 10% test split.")
    print(f"         Using all {len(df_exclusive)} exclusive molecules for test set.")
    test_size = len(df_exclusive)


# Calculate test proportion relative to exclusive molecules only
test_prop = test_size / len(df_exclusive) if len(df_exclusive) > 0 else 0


# Perform stratified split on exclusive molecules using Outcome column when available
if 'Outcome' in df_exclusive.columns:
    stratify_col = 'Outcome'
    print(f"\nPerforming stratified split on column: {stratify_col}")
    df_train_exclusive, df_test = train_test_split(
        df_exclusive, 
        test_size=test_prop,
        stratify=df_exclusive[stratify_col],
        random_state=42
)
else:
    print("\nNote: 'Outcome' column not found. Performing random split.")
    print("      For stratified split, ensure dataset contains 'Outcome' column.")
    df_train_exclusive, df_test = train_test_split(
        df_exclusive,
        test_size=test_prop,
        random_state=42
)


# Construct final training set: exclusive train molecules + all overlapping molecules
df_train = pd.concat([df_train_exclusive, df_overlapping], ignore_index=True)


print(f"\nFinal Dataset Split:")
print(f"  Training set (90%): {len(df_train)} molecules")
print(f"  Test set (10%): {len(df_test)} molecules")
print(f"  Actual split ratio: {len(df_train)/total_size*100:.1f}% train / {len(df_test)/total_size*100:.1f}% test")


# Validation: ensure no test set leakage into benchmark dataset
test_overlap_check = df_test['InChIKey'].isin(inchikey_benchmark).sum()
print(f"\nBenchmark Leakage Validation:")
print(f"  Test molecules found in benchmark: {test_overlap_check}")


if test_overlap_check == 0:
    print("  Status: PASSED - No test set leakage detected")
else:
    print("  Status: FAILED - Test set contains molecules from benchmark dataset")


# Export train-test splits (optional)
df_train.to_csv('train_dataset.csv', index=False)
df_test.to_csv('test_dataset.csv', index=False)



Dataset Partitioning Analysis:
  Total molecules in our dataset: 4212
  Exclusive molecules (not in benchmark): 1020
  Overlapping molecules (present in benchmark): 3192

Train-Test Split Configuration:
  Target test set size (10%): 421 molecules
  Exclusive molecules available: 1020

Performing stratified split on column: Outcome

Final Dataset Split:
  Training set (90%): 3791 molecules
  Test set (10%): 421 molecules
  Actual split ratio: 90.0% train / 10.0% test

Benchmark Leakage Validation:
  Test molecules found in benchmark: 0
  Status: PASSED - No test set leakage detected
